# Pattern 07: Contextual retrieval (Anthropic)

Follows this repo's mandatory 8-section notebook template -- this is one of "the 10 patterns."

**This notebook's committed execution uses `RAG_RECIPES_LLM=mock`** for the embedding step, the
contextualization step (Anthropic), and the final-answer generation step (OpenAI).

**What's genuinely real below, and what isn't:** the cost comparison in the cell after section 5
(with-caching vs. without-caching dollar figures) uses `recipes.pricing`'s actual verified
`claude-sonnet-5` rates, AND `MockLLM` deterministically simulates real cache write/read behavior
(the first call for a given paper's document text is billed as a cache write, every later chunk of
that same paper is billed as a cache read) -- so the WITH/WITHOUT split you see below is genuinely
differential, not just two calls to the same formula on identical numbers. What's still not real:
the absolute token counts are `MockLLM`'s pseudo-counts (word-length-based), not a real tokenizer or
API call, so the dollar magnitude will differ under a real key even though the mechanism and its
direction (caching is cheaper) are already demonstrated correctly. Section 7 (retrieval-quality
findings vs. pattern 01) is genuinely PENDING -- that needs real embeddings and a real Anthropic key.


## Reproducibility header

In [1]:
import platform
import subprocess
import sys

import numpy
import openai

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


platform: Windows-11-10.0.26200-SP0
python: 3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
openai sdk: 2.53.0
numpy: 2.5.2
git commit: 359a825ee7b5044969f2de3f3746e79b3f067638


## Setup (loaded once, used by every section below)

In [2]:
import os

os.environ.setdefault("RAG_RECIPES_LLM", "mock")

from evals.run import load_corpus_by_id, load_qa_set, run_pattern
from recipes.llm import MockLLM, get_llm

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")
llm = get_llm()  # used for generation (recipe_fn's own LLM calls)

# Judging needs its own backend: under a real API key this is the same
# real model, but under mock, `llm`'s canned generation text isn't valid
# JSON, and the judge prompts require JSON output. A separate MockLLM
# here demonstrates a clean, illustrative run instead of every question
# correctly (but noisily) failing to parse -- see evals/judges.py's
# JudgeParseError and evals/run.py's per-question error isolation.
if os.environ.get("RAG_RECIPES_LLM", "openai").lower() == "mock":
    judge_llm = MockLLM(default_response='{"score": 1, "reasoning": "Mock judge: looks fine."}')
else:
    judge_llm = llm

from recipes.embeddings import get_embedder
from recipes.llm import get_anthropic_llm

embedder = get_embedder()
anthropic_llm = get_anthropic_llm()
cost_log = []


## 1. What this pattern does

Contextual retrieval is a one-time indexing-cost preprocessing step: before embedding each corpus
chunk, an LLM call (`claude-sonnet-5`, `recipes/contextual.py`) reads the chunk's FULL source
document and writes a short (1-2 sentence) blurb situating that chunk within the paper. The blurb is
prepended to the chunk's text before embedding. Retrieval and final-answer generation are otherwise
identical to pattern 01 (naive dense) -- this pattern's entire novelty is in what gets embedded, not
in how retrieval or generation work.


## 2. When to use it

- Your chunks lose important meaning when read in isolation (e.g. "the model described above" with
  no antecedent in the chunk itself)
- You have a fixed, mostly-static corpus where the one-time indexing cost is amortized over many
  future queries
- You can afford the extra LLM-call cost per chunk at index-build time (see the cost comparison
  below for why prompt caching makes this economically viable)


## 3. When NOT to use it

- Your corpus changes frequently -- every update means re-running the (paid) contextualization step
- Your documents are already small/short enough that chunks rarely lose meaning out of context
- The document isn't large enough to make prompt caching kick in (the minimum cacheable prefix
  differs by model; see `recipes/contextual.py`'s module docstring for why this pattern uses
  `claude-sonnet-5` specifically rather than a cheaper Haiku-tier model)


## 4. Implementation

In [3]:
from recipes.contextual import make_retrieve_and_answer

retrieve_and_answer = make_retrieve_and_answer(
    corpus_by_id, embedder=embedder, llm=llm, anthropic_llm=anthropic_llm, cost_log=cost_log
)

# Try it on one question directly.
sample = retrieve_and_answer("What does PCEval stand for?", k=3)
print("retrieved:", sample.retrieved_chunk_ids)
print("answer:", sample.answer)


retrieved: ['arxiv:2601.00121#1', 'arxiv:2601.00130#2', 'arxiv:2601.00097#0']
answer: This is a mock response.


## 5. Run on our eval set

In [4]:
pattern_fn = retrieve_and_answer  # already built in section 4 -- avoid re-contextualizing the corpus

result = run_pattern(
    recipe_fn=pattern_fn,
    qa_set=qa_set,
    corpus_by_id=corpus_by_id,
    llm=judge_llm,
    pattern_name="07_contextual",
    judges_enabled=True,
)


=== 07_contextual (n=18) ===
  hit@3: 0.111  [95% CI 0.000, 0.278]
  hit@10: 0.278  [95% CI 0.111, 0.500]
  mrr: 0.110  [95% CI 0.017, 0.233]
  faithfulness: 1.000  [95% CI 1.000, 1.000]
  answer_relevance: 1.000  [95% CI 1.000, 1.000]
  citation_accuracy: 1.000  [95% CI 1.000, 1.000]
  filter_accuracy: 0.000  [95% CI 0.000, 0.000]
  p50_latency_ms: 0.3
  p95_latency_ms: 0.4
  usd_per_query: $0.00185
  eval_usd: $0.0333


## Cost comparison: with vs. without prompt caching

This pattern's leaderboard cost column must show both cached and uncached numbers, since Anthropic's contextual retrieval is only economically viable WITH prompt
caching. This uses `cost_log` (populated during section 4's `make_retrieve_and_answer` call) and the
real `recipes.pricing.cost_usd()` formula -- genuine arithmetic, even though the token volumes are
`MockLLM`'s pseudo-counts rather than a real API call's.

In [5]:
from recipes.pricing import cost_usd

total_input = sum(e["input_tokens"] for e in cost_log)
total_output = sum(e["output_tokens"] for e in cost_log)
total_cached = sum(e["cached_input_tokens"] for e in cost_log)
total_creation = sum(e["cache_creation_input_tokens"] for e in cost_log)

cost_with_caching = cost_usd(
    "claude-sonnet-5",
    input_tokens=total_input,
    output_tokens=total_output,
    cached_input_tokens=total_cached,
    cache_creation_input_tokens=total_creation,
)
# Counterfactual: same total tokens, but as if none of them ever hit a
# cache tier -- everything billed at the plain input rate.
cost_without_caching = cost_usd("claude-sonnet-5", input_tokens=total_input, output_tokens=total_output)

print(f"contextualization calls: {len(cost_log)}")
print(f"total input tokens: {total_input} (cached read: {total_cached}, cache write: {total_creation})")
print(f"cost WITH caching:    ${cost_with_caching:.6f}")
print(f"cost WITHOUT caching: ${cost_without_caching:.6f}")
print(f"savings: ${cost_without_caching - cost_with_caching:.6f} ({(1 - cost_with_caching / cost_without_caching) * 100:.1f}%)" if cost_without_caching > 0 else "")


contextualization calls: 54
total input tokens: 106514 (cached read: 54116, cache write: 27058)
cost WITH caching:    $0.132388
cost WITHOUT caching: $0.216268
savings: $0.083880 (38.8%)


## 6. Example query walkthrough

One example per eval-set category, showing retrieved chunks (from the CONTEXTUALIZED index) and the
(mocked) final answer. Under `MockEmbedder`, retrieval is deterministic-but-arbitrary (hash-based),
so which chunks come back is not meaningful -- only that the two-stage (contextualize, then embed)
code path runs end to end.

In [6]:
examples = {
    "keyword": "What does PCEval stand for?",
    "paraphrase": "Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?",
    "multi_hop": "The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?",
    "filter": "Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?",
}

for category, question in examples.items():
    result = retrieve_and_answer(question, k=3)
    print(f"--- {category} ---")
    print(f"Q: {question}")
    print(f"Retrieved: {result.retrieved_chunk_ids}")
    print(f"A: {result.answer}")
    print()


--- keyword ---
Q: What does PCEval stand for?
Retrieved: ['arxiv:2601.00121#1', 'arxiv:2601.00130#2', 'arxiv:2601.00097#0']
A: This is a mock response.

--- paraphrase ---
Q: Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?
Retrieved: ['arxiv:2601.00087#0', 'arxiv:2601.00097#0', 'arxiv:2601.00116#2']
A: This is a mock response.

--- multi_hop ---
Q: The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?
Retrieved: ['arxiv:2601.00116#1', 'arxiv:2601.00090#2', 'arxiv:2601.00087#2']
A: This is a mock response.

--- filter ---
Q: Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?
Retrieved: ['arxiv:2601.00087#2', 'arxiv:2601.11580#2', 'arxiv:2601.00090#0']
A: This is a mock response.



## 7. Where this pattern FAILS

**PENDING: real findings from a one-off real-embeddings run.** A real `OPENAI_API_KEY` was not yet
available in the environment when this notebook was authored. This section will be replaced with a
static table of genuine hit@k/mrr failures (matching the format used in `02_bm25.ipynb`/
`04_rerank.ipynb` section 7), computed via a real, uncommitted exploratory run once a key is
available. The claim will be labeled with the date it was run
and its actual dollar cost, and will not be presented as live-executed cell output, to avoid
implying a mock re-run reproduces it (see this notebook's top-of-file disclaimer).


## 8. Copy-paste snippet

Meant for pasting into your own project, not executed as a cell in this notebook.

```python
"""Minimal contextual retrieval + generation, no eval harness."""
from recipes.embeddings import get_embedder
from recipes.contextual import make_retrieve_and_answer
from recipes.llm import get_llm, get_anthropic_llm

corpus_by_id = {}  # {chunk_id: {"paper_id": ..., "text": ..., ...}, ...} -- fill in your own chunks
embedder = get_embedder()
llm = get_llm()
anthropic_llm = get_anthropic_llm()  # requires ANTHROPIC_API_KEY

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm, anthropic_llm=anthropic_llm)
result = retrieve_and_answer("your question here", k=5)
print(result.answer)
```
